In [4]:
# =============================================================================
# ENGENHARIA DE FEATURES + FILTRO AUTOMÁTICO DE REDUNDÂNCIA (THRESHOLD 0.8)
# =============================================================================
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

print("📥 1. Carregando os Dados Originais...")
try:
    df = pd.read_csv('dataset_limpo.csv') 
    df = df.sort_values(['ID_Trem', 'Ciclo']).reset_index(drop=True)
    if df['ID_Trem'].dtype == 'object':
        df['ID_Trem'] = df['ID_Trem'].astype(str).str.extract('(\d+)').astype(int)
except FileNotFoundError:
    print("❌ ERRO: O arquivo 'dataset_limpo.csv' não foi encontrado!")
    raise

print("✂️ 2. Aplicando a Divisão Estrita (30 Treino / 18 Validação)...")
trens_treino_ids = [2, 4, 5, 6, 8, 10, 11, 14, 15, 16, 18, 19, 22, 23, 24, 
                    26, 29, 30, 31, 32, 33, 34, 35, 36, 39, 41, 42, 43, 44, 45]
todos_trens = list(range(1, 49))
trens_val_ids = [t for t in todos_trens if t not in trens_treino_ids]

df['Grupo'] = np.where(df['ID_Trem'].isin(trens_treino_ids), 'Treino', 
              np.where(df['ID_Trem'].isin(trens_val_ids), 'Validação', 'Descartado'))
df = df[df['Grupo'] != 'Descartado'].copy()

print("🧬 3. Injetando Engenharia Profunda de Features (Irreversibilidade)...")
df['Vib_Energia_MaxAcumulado'] = df.groupby('ID_Trem')['Vibracao_Energia'].cummax()
df['Vib_Max_MaxAcumulado'] = df.groupby('ID_Trem')['Vibracao_Max'].cummax()

df['Aceleracao_Energia'] = df.groupby('ID_Trem')['Vibracao_Energia'].diff(3).fillna(0)
df['Aceleracao_Energia'] = np.clip(df['Aceleracao_Energia'], 0, None) 

if 'Distancia_Ref_Atual' in df.columns:
    df['Dist_Ref_Suavizada'] = df.groupby('ID_Trem')['Distancia_Ref_Atual'].transform(lambda x: x.rolling(10, min_periods=1).mean())
    df = df.drop(columns=['Distancia_Ref_Atual'])

print("🔪 4. Aplicando a Guilhotina de Redundância (|Correlação| > 0.8)...")
cols_ignorar = ['ID_Trem', 'Ciclo', 'Grupo', 'RUL_Alvo']
features_iniciais = [c for c in df.columns if c not in cols_ignorar]

# Reordenamos para garantir que as features recém-criadas (Pró-PH) tenham prioridade de sobrevivência
features_prioridade = ['Dist_Ref_Suavizada', 'Vib_Energia_MaxAcumulado', 'Vib_Max_MaxAcumulado', 'Aceleracao_Energia']
outras_features = [f for f in features_iniciais if f not in features_prioridade]
features_ordenadas = features_prioridade + outras_features

# Calcula a matriz de correlação absoluta
matriz_corr_abs = df[features_ordenadas].corr().abs()

# Seleciona apenas a parte superior do triângulo da matriz para não duplicar comparações
upper_tri = matriz_corr_abs.where(np.triu(np.ones(matriz_corr_abs.shape), k=1).astype(bool))

# Encontra colunas onde qualquer valor seja maior que 0.80
features_redundantes = [coluna for coluna in upper_tri.columns if any(upper_tri[coluna] > 0.80)]

# Remove do dataframe
df = df.drop(columns=features_redundantes)
features_finais = [c for c in df.columns if c not in cols_ignorar]

print(f"🗑️ Foram eliminadas {len(features_redundantes)} features clonadas/redundantes:")
for f in features_redundantes:
    print(f"   - {f}")

print("\n🔍 5. Gerando o Novo Mapa de Calor (Pós-Expurgo)...")
matriz_corr_final = df[features_finais].corr().round(2)

fig = go.Figure(data=go.Heatmap(
    z=matriz_corr_final.values,
    x=matriz_corr_final.columns,
    y=matriz_corr_final.columns,
    colorscale='RdBu_r', 
    zmin=-1, zmax=1,
    text=matriz_corr_final.values,
    texttemplate="%{text}",
    hoverinfo="text"
))

fig.update_layout(
    title="<b>✨ Raio-X Pós-Expurgo (Apenas Features Independentes)</b>",
    title_x=0.5,
    width=900, height=800,
    xaxis=dict(tickangle=45)
)
fig.show(renderer='browser')

print("💾 6. Exportando o Novo Dataset de Batalha Blindado...")
nome_arquivo = 'dataset_features_ph_otimizado.csv'
df.to_csv(nome_arquivo, index=False)
print(f"🚀 Sucesso! Arquivo '{nome_arquivo}' gerado com {len(features_finais)} features puras e independentes.")

📥 1. Carregando os Dados Originais...
✂️ 2. Aplicando a Divisão Estrita (30 Treino / 18 Validação)...
🧬 3. Injetando Engenharia Profunda de Features (Irreversibilidade)...
🔪 4. Aplicando a Guilhotina de Redundância (|Correlação| > 0.8)...
🗑️ Foram eliminadas 7 features clonadas/redundantes:
   - Aceleracao_Energia
   - Vibracao_Energia
   - MA5_Vibracao_Max
   - MA5_Vibracao_Energia
   - MA5_Vibracao_Std
   - Delta_Vibracao_Std
   - Acel_Vibracao_Std

🔍 5. Gerando o Novo Mapa de Calor (Pós-Expurgo)...
💾 6. Exportando o Novo Dataset de Batalha Blindado...
🚀 Sucesso! Arquivo 'dataset_features_ph_otimizado.csv' gerado com 6 features puras e independentes.
